# Hawkes 2-Point Function — Full Pipeline Demo

**Model:** Nonlinear Hawkes process, 2 populations, delta-function synaptic filters, quadratic nonlinearity

**Observable:** 2-point correlation function $\langle \delta\dot{n}_1(t_1)\, \delta\dot{n}_2(t_2) \rangle$

**Loop levels:** Tree-level ($\ell = 0$) and 1-loop ($\ell = 1$)

---

## What this notebook does

This notebook walks through the full automated Feynman diagram pipeline (Phases A–G) for a concrete MSRJD field theory.

### Section 1 — Theory / Model Information
Loads the 2-population nonlinear Hawkes model and prints all of its defining data: response fields ($\tilde{n}_i$), physical fluctuation fields ($\delta\dot{n}_i$), parameters, synaptic kernels, operators, and the nonlinear transfer function $\phi$. Then expands the MSRJD action to the required Taylor order and extracts the free-action kernel matrix $\mathbf{K}$, its Fourier transform $\hat{\mathbf{K}}(\omega)$, the retarded propagator $\hat{\mathbf{G}}(\omega) = \hat{\mathbf{K}}^{-1}(\omega)$, its poles, residue matrices, and the time-domain propagator $\mathbf{G}(t)$.

### Section 2 — Prediagram Enumeration
For the 2-point function ($k=2$), enumerates all labeled trees, undirected topologies, and directed prediagrams at tree level ($\ell=0$) and 1-loop ($\ell=1$). Plots each topology and prediagram with color-coded vertices: black = external legs, red = source vertices (noise insertions, in-degree 0), light blue = internal interaction vertices.

### Section 3 — Vertex Extraction
Reads off all interaction vertex types (`VertexType`) and noise source types (`SourceType`) from the expanded action. Each vertex type has a bigrade $(n_{\tilde{}}, n_{\text{phys}})$, a list of response and physical legs, and a coefficient. Checks that the current Taylor order is sufficient to cover the maximum vertex degree appearing in the prediagrams.

### Section 4 — Prediagram Filtering
Discards any prediagram whose internal vertex degree signatures don't match any available vertex or source type. Reports how many prediagrams survive at each loop level and plots the survivors.

### Section 5 — Type Assignment (Fully Labeled Diagrams)
Sets up the external fields for the 2-point function ($\delta\dot{n}_1$, $\delta\dot{n}_2$) and builds the field-to-matrix-index maps. Then runs the constraint-satisfaction engine on each surviving prediagram: for every valid assignment of vertex types to internal vertices and field types to edges (consistent with the propagator matrix $\hat{\mathbf{G}}$), it produces a `TypedDiagram`. Each typed diagram is printed showing its external leg assignments, vertex types with coefficients, and edge propagator labels $\hat{G}_{ij}$.

### Section 6 — Diagram Isomorphism Check
Groups the enumerated typed diagrams into equivalence classes by canonical signature. Two diagrams are isomorphic if they have identical external leg assignments, vertex types, and propagator indices on every edge. Reports the multiplicity of each unique diagram — this multiplicity is the symmetry factor that cancels the $1/n!$ from the Taylor expansion coefficients.

In [ ]:
%display latex
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from IPython.display import display, Math
from sage.all import (
    SR, matrix, latex, I, pi, exp, diff, solve, integrate, oo,
    dirac_delta, function, var
)

from msrjd.core.field_theory import FieldTheory, fourier_transform, inverse_fourier_transform
from msrjd.core.vertices import extract_vertex_types, extract_source_types, available_degrees
from msrjd.enumeration.loop_diagram_enumeration import enumerate_all
from msrjd.diagrams.filter import filter_prediagrams, classify_prediagram_vertices
from msrjd.diagrams.type_assignment import (
    enumerate_typed_diagrams, enumerate_all as enumerate_all_typed,
    build_field_index_map, TypedDiagram
)
from msrjd.diagrams.causality import filter_causal
from msrjd.diagrams.symmetry import symmetry_factor, compute_all_symmetry_factors

from models.hawkes_sage import HAWKES_MODEL

print('Imports loaded.')

---
## 1. Theory / Model Information

In [ ]:
m = HAWKES_MODEL
print(f"Model:  {m['name']}")
print(f"Convention: {m['background_rate_convention']}")
print()

print('── Index sets ──')
for name, vals in m['index_sets'].items():
    print(f'  {name}: {list(vals)}')
print()

print('── Response fields (not expanded — full integration variables) ──')
for f in m['response_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Physical fields (expanded around MF background) ──')
for f in m['physical_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Parameters ──')
for p in m.get('parameters', []):
    idx = '(indexed)' if p.get('indexed') else '(scalar)'
    dom = f", domain={p['domain']}" if p.get('domain') else ''
    print(f"  {p['name']}  {idx}{dom}  — {p.get('description', '')}")
print()

print('── Kernels ──')
for k in m.get('kernels', []):
    print(f"  {k['name']}  — {k.get('description', '')}")
print()

print('── Operators ──')
for o in m.get('operators', []):
    print(f"  {o['name']}  (latex: ${o.get('latex_name', o['name'])}$)  — {o.get('description', '')}")
print()

print('── Nonlinear functions ──')
for f in m.get('functions', []):
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Specializations ──')
print('  φ quadratic (cubic and higher coefficients = 0)')
print('  g = δ(t)  (instantaneous synaptic coupling)')

### 1.1 Expand the action and show all sectors

In [ ]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print(f'Taylor order: {ft.taylor_order}')
print(f'Polynomial ring: {R}')
print(f'Ring generators: {R.gens()}')
print(f'Response generators (n_tilde={ft._n_tilde}): {R.gens()[:ft._n_tilde]}')
print(f'Physical generators: {R.gens()[ft._n_tilde:]}')
print()

passed = ft.sanity_check()
print()
ft.summary()

### 1.2 Propagator

Extract the kernel matrix $\mathbf{K}$ from the free action, Fourier transform, and invert.

In [ ]:
import signal

S_free = ft.free_action()
ring_gen_names = [str(g) for g in R.gens()]

resp_names = [f'vt{i+1}' for i in ns.pop] + [f'nt{i+1}' for i in ns.pop]
phys_names = [f'dv{i+1}' for i in ns.pop] + [f'dn{i+1}' for i in ns.pop]

pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

nf = len(resp_names)
K_data = [[SR(0)] * nf for _ in range(nf)]
for exp_vec, coeff in S_free.dict().items():
    row = col = None
    for k in range(len(ring_gen_names)):
        if exp_vec[k] > 0:
            if k in pos_to_row: row = pos_to_row[k]
            if k in pos_to_col: col = pos_to_col[k]
    if row is not None and col is not None:
        K_data[row][col] += SR(coeff)

K_mat = matrix(SR, K_data)

# Convert to kernel form
Dt       = ns.Dt
delta_D  = ns.delta_D
delta_Dp = ns.delta_Dp

def _to_kernel(c):
    c = SR(c)
    if c.has(delta_D) or c.has(ns.g):
        return c
    p0   = c.subs({Dt: 0})
    rest = (c - p0).subs({Dt: delta_Dp})
    return p0 * delta_D + rest

K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

resp_sr  = [ns.vt[i] for i in ns.pop] + [ns.nt[i] for i in ns.pop]
phys_sr  = [ns.dv[i] for i in ns.pop] + [ns.dn[i] for i in ns.pop]

print('Field ordering:')
display(Math(r'\tilde{\mathbf{a}} = ' + latex(vector(resp_sr))
             + r',\qquad \mathbf{a} = ' + latex(vector(phys_sr))))
print()
print('Kernel matrix K (time domain):')
display(Math(r'\mathbf{K} = ' + latex(K_ker)))

# Fourier transform
t_var = SR.var('t')
omega = SR.var('omega', latex_name=r'\omega')

time_domain = {
    delta_D:  dirac_delta(t_var),
    delta_Dp: diff(dirac_delta(t_var), t_var),
    ns.g:     function('g')(t_var),
}

K_ft_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        c = K_ker[i, j]
        if not c.is_zero():
            K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

K_ft = matrix(SR, K_ft_data)
print('Fourier-domain kernel:')
display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

# Propagator inverse
G_ft = K_ft.inverse().apply_map(lambda e: e.factor())
G_ft_explicit = True
print('Propagator:')
display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))

# Adjugate, determinant, poles
adj_ft  = K_ft.adjugate()
D_omega = K_ft.det().expand()
D_prime = diff(D_omega, omega)

pole_eqs = solve(D_omega == 0, omega)
pole_vals = [eq.rhs() if hasattr(eq, 'rhs') else eq for eq in pole_eqs]

print(f'\ndet(K̂) = {D_omega}')
print(f'Poles ({len(pole_vals)}):')
for k, p in enumerate(pole_vals):
    display(Math(r'\omega_{' + str(k+1) + '} = ' + latex(p)))

# Residue matrices
C_mats = []
for omega_k in pole_vals:
    C_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            n_ij = adj_ft[i, j]
            if not n_ij.is_zero():
                num = n_ij.subs({omega: omega_k})
                den = D_prime.subs({omega: omega_k})
                C_data[i][j] = (I * num / den).factor()
    C_mats.append(matrix(SR, C_data))

print('Residue matrices:')
for k, C in enumerate(C_mats):
    display(Math(r'\mathbf{C}_{' + str(k+1) + r'} = ' + latex(C)))

# Time-domain propagator
G_t = sum(C_mats[k] * exp(I * pole_vals[k] * t_var) for k in range(len(pole_vals)))
G_t = G_t.apply_map(lambda e: e.simplify_full())
print('Time-domain propagator G(t) for t > 0:')
display(Math(r'\mathbf{G}(t) = ' + latex(G_t)))

---
## 2. Prediagram Enumeration

Enumerate all trees, topologies, and prediagrams for the 2-point function at tree level ($\ell=0$) and 1-loop ($\ell=1$).

**Color code:** black = external legs, red = source vertices ($\mathrm{in}=0$), light blue = internal interaction vertices

In [ ]:
from sage.plot.plot import graphics_array

def plot_prediagrams(pds, title_prefix=''):
    """Plot prediagrams with colored vertices."""
    if not pds:
        print('  (none)')
        return
    plots = []
    for i, (D, G, leaves, internal) in enumerate(pds):
        leaves_set = set(leaves)
        color_map = {}
        for v in D.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            elif D.in_degree(v) == 0:
                color_map.setdefault('red', []).append(v)
            else:
                color_map.setdefault('lightblue', []).append(v)
        plots.append(D.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

def plot_topologies(topos, title_prefix=''):
    """Plot undirected topologies."""
    if not topos:
        print('  (none)')
        return
    plots = []
    for i, (G, leaves, internal) in enumerate(topos):
        leaves_set = set(leaves)
        color_map = {}
        for v in G.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            else:
                color_map.setdefault('lightgray', []).append(v)
        plots.append(G.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

print('Plot helpers defined.')

### 2.1 Tree level ($\ell = 0$)

In [ ]:
k = 2

trees_0, topos_0, pds_0, counts_0 = enumerate_all(k, 0, verbose=False)
print(f'k={k}, ell=0:  {counts_0["n_trees"]} trees,  {counts_0["n_topologies"]} topologies,  {counts_0["n_prediagrams"]} prediagrams')
print()

print('── Trees ──')
for i, (T, j, nl) in enumerate(trees_0):
    print(f'  Tree {i+1}: {T.num_verts()} vertices, {T.num_edges()} edges, {nl} leaves, j={j}')

print()
print('── Topologies ──')
plot_topologies(topos_0, title_prefix='Topo ')

print('── Prediagrams ──')
plot_prediagrams(pds_0, title_prefix='PD ')

for i, (D, G, leaves, internal) in enumerate(pds_0):
    print(f'  PD {i+1}: edges={D.edges(labels=False)}')
    for v in sorted(D.vertices()):
        role = 'leaf' if v in leaves else ('source' if D.in_degree(v)==0 else 'internal')
        print(f'    v{v}: in={D.in_degree(v)}, out={D.out_degree(v)} ({role})')

### 2.2 One loop ($\ell = 1$)

In [ ]:
trees_1, topos_1, pds_1, counts_1 = enumerate_all(k, 1, verbose=False)
print(f'k={k}, ell=1:  {counts_1["n_trees"]} trees,  {counts_1["n_topologies"]} topologies,  {counts_1["n_prediagrams"]} prediagrams')
print()

print('── Topologies ──')
plot_topologies(topos_1, title_prefix='Topo ')

print('── Prediagrams ──')
plot_prediagrams(pds_1, title_prefix='PD ')

for i, (D, G, leaves, internal) in enumerate(pds_1):
    print(f'  PD {i+1}: edges={D.edges(labels=False)}')
    for v in sorted(D.vertices()):
        role = 'leaf' if v in leaves else ('source' if D.in_degree(v)==0 else 'internal')
        print(f'    v{v}: in={D.in_degree(v)}, out={D.out_degree(v)} ({role})')

---
## 3. Vertex Extraction

Extract typed vertices (`VertexType`, `SourceType`) from the expanded action, based on the maximum vertex degree required by the prediagrams.

In [ ]:
from msrjd.enumeration.degree_scan import max_vertex_degree, scan_source_vertices

all_pds = pds_0 + pds_1
max_deg = max_vertex_degree(all_pds)
src_degs = scan_source_vertices(all_pds)
print(f'Max vertex degree across all prediagrams: {max_deg}')
print(f'Source vertex out-degrees needed: {src_degs}')
print(f'Current Taylor order: {ft.taylor_order}')
print(f'Sufficient: {ft.taylor_order >= max_deg}')
print()

vtypes = extract_vertex_types(ft)
stypes = extract_source_types(ft)

print(f'── Interaction vertex types ({len(vtypes)}) ──')
for i, vt in enumerate(vtypes):
    print(f'  V{i+1}: bigrade={vt.bigrade}, in_deg={vt.in_degree}, out_deg={vt.out_degree}')
    print(f'        resp_legs={vt.response_legs}, phys_legs={vt.physical_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(vt.coefficient)}'))

print(f'\n── Source types ({len(stypes)}) ──')
for i, st in enumerate(stypes):
    print(f'  S{i+1}: bigrade={st.bigrade}, out_deg={st.out_degree}')
    print(f'        resp_legs={st.response_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(st.coefficient)}'))

int_degs, src_degs_avail = available_degrees(vtypes, stypes)
print(f'\nAvailable interaction degrees (in, out): {sorted(int_degs)}')
print(f'Available source out-degrees: {sorted(src_degs_avail)}')

---
## 4. Prediagram Filtering

Remove prediagrams whose vertex degree signatures don't match any available vertex or source type.

In [ ]:
print('── Tree level (ell=0) ──')
kept_0, disc_0 = filter_prediagrams(pds_0, vtypes, stypes)
print(f'  {len(pds_0)} prediagrams → {len(kept_0)} kept, {disc_0} discarded')

print()
print('── 1-loop (ell=1) ──')
kept_1, disc_1 = filter_prediagrams(pds_1, vtypes, stypes)
print(f'  {len(pds_1)} prediagrams → {len(kept_1)} kept, {disc_1} discarded')

if disc_1 > 0:
    print(f'\n  Discarded prediagram indices:')
    kept_set = set(id(p) for p in kept_1)
    for i, pd in enumerate(pds_1):
        if id(pd) not in kept_set:
            D = pd[0]
            degs = [(D.in_degree(v), D.out_degree(v)) for v in pd[3]]  # internal verts
            print(f'    PD {i+1}: internal vertex degrees = {degs}')

print()
print('── Surviving prediagrams (tree level) ──')
plot_prediagrams(kept_0, title_prefix='Tree PD ')

print('── Surviving prediagrams (1-loop) ──')
plot_prediagrams(kept_1, title_prefix='1L PD ')

---
## 5. Type Assignment — Fully Labeled Diagrams

Enumerate all valid field-type assignments on each surviving prediagram.

**External legs:** $\delta\dot{n}_1$ and $\delta\dot{n}_2$ (the 2-point function fields).  
**Edge convention:** each directed edge $u \to v$ carries propagator $\hat{G}_{ij}(\omega)$ where $i$ is a response-field index from $u$ and $j$ is a physical-field index from $v$.

In [ ]:
# External fields for the 2-point function <dn1 dn2>
# dn1, dn2 are physical fields — they sit at the incoming-edge end of leaves
external_fields = [('dn', 1), ('dn', 2)]

# Build field index maps
ring_var_names_list = list(ns._ring_var_names)
n_tilde = ft._n_tilde
resp_idx, phys_idx = build_field_index_map(ring_var_names_list, n_tilde)

print('Response field index map:')
for leg, idx in sorted(resp_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} → row {idx}')
print()
print('Physical field index map:')
for leg, idx in sorted(phys_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} → col {idx}')

# Helper to display typed diagrams nicely
def show_typed_diagram(td, idx):
    D = td.prediagram[0]
    leaves = td.prediagram[2]
    print(f'\n{"="*60}')
    print(f'Diagram {idx}')
    print(f'{"="*60}')
    
    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} ← {field[0]}{field[1]}')
    
    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            display(Math(f'\\quad\\text{{coeff}} = {latex(vtype.coefficient)}'))
    
    print('  Edges (propagator assignments):')
    for (u, v), (resp_leg, phys_leg) in sorted(td.edge_types.items()):
        ri, pi = td.propagator_indices.get((u, v), ('?', '?'))
        print(f'    {u} → {v}:  {resp_leg[0]}{resp_leg[1]} → {phys_leg[0]}{phys_leg[1]}  '
              f'[G_hat[{ri},{pi}]]')

### 5.1 Tree-level typed diagrams

In [ ]:
typed_tree = enumerate_all_typed(
    kept_0, external_fields, vtypes, stypes, G_ft, resp_idx, phys_idx
)
print(f'Tree-level: {len(kept_0)} prediagrams → {len(typed_tree)} typed diagrams')

for i, td in enumerate(typed_tree, 1):
    show_typed_diagram(td, f'Tree-{i}')

### 5.2 One-loop typed diagrams

In [ ]:
typed_1loop = enumerate_all_typed(
    kept_1, external_fields, vtypes, stypes, G_ft, resp_idx, phys_idx
)
print(f'1-loop: {len(kept_1)} prediagrams → {len(typed_1loop)} typed diagrams')

for i, td in enumerate(typed_1loop, 1):
    show_typed_diagram(td, f'1L-{i}')

---
## 6. Diagram Isomorphism Check

Identify which typed diagrams are truly distinct. Two typed diagrams are **isomorphic** if they have the same:
- External leg assignments (which field at each leaf)
- Vertex type at each internal vertex (same coefficient, same legs)
- Propagator assignment on every edge (same $\hat{G}_{ij}$ indices)

We build a canonical signature for each diagram and group duplicates into equivalence classes.

In [ ]:
from collections import defaultdict

def diagram_signature(td):
    """
    Build a hashable canonical signature for a typed diagram.
    
    Two diagrams with the same signature are identical up to the
    internal ordering of leg permutations — they represent the
    same Feynman diagram.
    """
    # External legs: sorted (leaf, field) pairs
    ext = tuple(sorted(td.external_legs.items()))
    
    # Vertex assignments: sorted (vertex, type_key) pairs
    verts = []
    for v, vtype in sorted(td.vertex_assignments.items()):
        tname = type(vtype).__name__
        resp = tuple(vtype.response_legs)
        phys = tuple(vtype.physical_legs) if hasattr(vtype, 'physical_legs') else ()
        verts.append((v, tname, str(vtype.coefficient), vtype.bigrade, resp, phys))
    verts = tuple(verts)
    
    # Edge propagator assignments: sorted (edge, prop_indices) pairs
    edges = tuple(sorted(
        ((u, v), td.propagator_indices[(u, v)])
        for (u, v) in td.edge_types
    ))
    
    return (ext, verts, edges)


def find_equivalence_classes(typed_diagrams, label_prefix=''):
    """Group typed diagrams by their canonical signature."""
    sig_to_indices = defaultdict(list)
    signatures = []
    for i, td in enumerate(typed_diagrams):
        sig = diagram_signature(td)
        signatures.append(sig)
        sig_to_indices[sig].append(i)
    
    classes = list(sig_to_indices.values())
    
    print(f'{len(typed_diagrams)} typed diagrams → {len(classes)} unique diagrams')
    print()
    for ci, members in enumerate(classes, 1):
        member_labels = [f'{label_prefix}{m+1}' for m in members]
        multiplicity = len(members)
        print(f'  Class {ci} (multiplicity {multiplicity}): diagrams {member_labels}')
        
        # Show the representative diagram
        td = typed_diagrams[members[0]]
        print(f'    External legs: ', end='')
        print(', '.join(f'leaf {lf}←{f[0]}{f[1]}' for lf, f in sorted(td.external_legs.items())))
        print(f'    Edges: ', end='')
        print(', '.join(
            f'{u}→{v} [G_hat[{td.propagator_indices[(u,v)][0]},{td.propagator_indices[(u,v)][1]}]]'
            for (u, v) in sorted(td.edge_types)
        ))
        if td.vertex_assignments:
            for v, vtype in sorted(td.vertex_assignments.items()):
                tname = type(vtype).__name__
                print(f'    v{v} ({tname}): resp={vtype.response_legs}', end='')
                if hasattr(vtype, 'physical_legs'):
                    print(f', phys={vtype.physical_legs}', end='')
                print(f', coeff={vtype.coefficient}')
    
    return classes


print('='*60)
print('TREE-LEVEL ISOMORPHISM ANALYSIS')
print('='*60)
tree_classes = find_equivalence_classes(typed_tree, label_prefix='Tree-')

print()
print('='*60)
print('1-LOOP ISOMORPHISM ANALYSIS')
print('='*60)
loop_classes = find_equivalence_classes(typed_1loop, label_prefix='1L-')